In [2]:
%load_ext autoreload
%autoreload 2
import eval
import os
import glob
import pickle

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


/data/cb/mihirb14/projects/BoltzDesign1/utils/geometry.py:8: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(False)
/data/cb/mihirb14/projects/BoltzDesign1/utils/geometry.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(False)
/data/cb/mihirb14/projects/BoltzDesign1/utils/geometry.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(False)


In [2]:
# out_dir = f"./out/onemotif_twostates_pos/"
# out_dir = f"/data/cb/mihirb14/projects/BoltzDesign1/out/test/zinc/onemotif_twostates_pos"

# out_dir = f"out/onemotif_twostates_neg_OQO/"
out_dir = f"out/onemotif_twostates_pos_EcoRIdna"
motif = "3ixt"
motif_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motif}.pdb"
motif_out_dir = os.path.join(out_dir,motif)

In [3]:


def motif_results(motif_out_dir, motif_pdb, motif):
    rows = []
    for design_dir in sorted(glob.glob(os.path.join(motif_out_dir, "design*"))):
        design_name = os.path.basename(design_dir)

        # load motif mask
        with open(os.path.join(design_dir, f"{motif}_spec.pkl"), "rb") as f:
            motif_mask = pickle.load(f)["motif_mask"]

        for state in [0, 1]:
            with open(os.path.join(design_dir, f"state{state}.pkl"), "rb") as f:
                outdict = pickle.load(f)

            for sample_idx in range(5):  # assume 5 samples per state
                pdb_file = os.path.join(design_dir, f"state{state}_sample{sample_idx}.pdb")
                if not os.path.exists(pdb_file):
                    continue

                rmsd = eval.motifRMSD(motif_pdb, pdb_file, motif_mask)

                rows.append({
                    "design": design_name,
                    "state": state,
                    "sample": sample_idx,
                    "motifrmsd": rmsd.item() if hasattr(rmsd, "item") else float(rmsd),
                    "plddt": outdict["plddt"].cpu().numpy()[sample_idx].mean(),
                    "ptm": outdict["ptm"].cpu().numpy()[sample_idx],
                })
                
    full = pd.DataFrame(rows)

    agg = full.groupby(["design", "state"]).agg(
        motifRMSD_mean=("motifrmsd", "mean"),
        motifRMSD_std=("motifrmsd", "std"),
        plddt=("plddt", "mean"),
        ptm=("ptm", "mean")
    ).reset_index()

    agg = agg.pivot(index="design", columns="state").reset_index()
    agg.columns = ["_".join(map(str, col)).rstrip("_") for col in agg.columns.to_flat_index()]
    agg = agg.rename(columns=lambda c: c.replace("_0", "_unbound").replace("_1", "_bound"))
    
    agg.to_csv(os.path.join(motif_out_dir,"_aggresults.csv"),index=False)
    full.to_csv(os.path.join(motif_out_dir,"_fullresults.csv"),index=False)


    return full, agg



df_full,df_agg = motif_results(motif_out_dir,motif_pdb, motif)
df_agg

,design,motifRMSD_mean_unbound,motifRMSD_mean_bound,motifRMSD_std_unbound,motifRMSD_std_bound,plddt_unbound,plddt_bound,ptm_unbound,ptm_bound
0,design0,4.664252,5.086625,0.782910,0.944184,0.590775,0.564516,0.252904,0.440575
1,design1,1.860876,4.059321,0.480045,0.260054,0.638880,0.717019,0.424371,0.654076
2,design10,5.411712,2.033903,1.931009,0.313449,0.662440,0.660536,0.347647,0.629355
3,design11,6.094125,5.145693,2.325308,0.605575,0.686711,0.581273,0.287605,0.459919
4,design12,5.593869,4.307042,2.160558,0.528610,0.582044,0.551407,0.218844,0.371662
...,...,...,...,...,...,...,...,...,...
95,design95,6.209118,3.490568,0.303835,0.081871,0.712388,0.641922,0.460747,0.573783
96,design96,8.242839,4.358994,1.645464,0.104949,0.706028,0.621336,0.288514,0.340918
97,design97,5.758945,5.007990,2.842360,0.152196,0.620248,0.589107,0.244637,0.488764
98,design98,2.045386,1.681087,0.701639,0.145253,0.646123,0.657174,0.377256,0.703398


In [5]:
len(df_agg[(df_agg["motifRMSD_mean_unbound"]>1.0) & (df_agg["motifRMSD_mean_bound"]<=1.0)]) # condition for one motif two states positive allostery

1

In [6]:


def plot_scatter_motifrmsd(df, motif):
    xrange = [0, 10]
    yrange = [0, 10]

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("", "with std ± 1")
    )

    fig.add_trace(
        go.Scatter(
            x=df["motifRMSD_mean_unbound"],
            y=df["motifRMSD_mean_bound"],
            mode="markers",
            marker=dict(
                color=df["plddt_unbound"],
                colorscale="sunsetdark",
                showscale=True,
                colorbar=dict(title="pLDDT",outlinewidth=0)
            ),
            text=df["design"],
            name="Designs"
        ),
        row=1, col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=df["motifRMSD_mean_unbound"],
            y=df["motifRMSD_mean_bound"],
            mode="markers",
            marker=dict(
                color=df["plddt_unbound"],
                colorscale="sunsetdark",
                showscale=False
            ),
            text=df["design"],
            error_x=dict(array=df["motifRMSD_std_unbound"], color="gray", thickness=1),
            error_y=dict(array=df["motifRMSD_std_bound"], color="gray", thickness=1),
            name="Designs (err)"
        ),
        row=1, col=2
    )

    for c in [1, 2]:
        fig.add_shape(
            type="line", x0=0, y0=0, x1=10, y1=10,
            line=dict(color="lightgray", dash="dash"),
            row=1, col=c
        )

    fig.update_xaxes(range=xrange, title="Unbound motif RMSD", row=1, col=1)
    fig.update_yaxes(range=yrange, title="Bound motif RMSD", row=1, col=1)
    fig.update_xaxes(range=xrange, title="Unbound motif RMSD", row=1, col=2)
    fig.update_yaxes(range=yrange, title="Bound motif RMSD", row=1, col=2)

    fig.update_layout(
        width=1200, height=600,
        title=f"({motif}) Unbound vs Bound motif RMSD",
        yaxis_scaleanchor="x",
        yaxis2_scaleanchor="x2",
        showlegend=False
    )

    return fig


plot_scatter_motifrmsd(df_agg, motif)


### onemotif_twostates_neg with sm ligand modulation

In [5]:
successes_neg = []
out_dir = f"out/onemotif_twostates_neg_OQO/"

for motif in sorted(os.listdir(out_dir)):
    motif_out_dir = os.path.join(out_dir, motif)
    
    agg_path = os.path.join(motif_out_dir, "_aggresults.csv")
    full_path = os.path.join(motif_out_dir, "_fullresults.csv")

    if not (os.path.exists(agg_path) and os.path.exists(full_path)):
        motif_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motif}.pdb"
        full, agg = motif_results(motif_out_dir, motif_pdb, motif)
    else:
        full = pd.read_csv(full_path)
        agg = pd.read_csv(agg_path)

    # condition for one motif two states negative allostery
    n_success = len(
        agg[(agg["motifRMSD_mean_unbound"] <= 1.0) 
            & (agg["motifRMSD_mean_bound"] > 1.0)  
            & ((agg["motifRMSD_mean_bound"] - agg["motifRMSD_mean_unbound"]).abs() >= 0.5) 
            & (agg["motifRMSD_std_unbound"] <= 0.5)
            & (agg["motifRMSD_std_bound"] <= 0.5)
            ]
    )

    successes_neg.append({
        "motif": motif,
        "effector":"OQO",
        "task": "onemotif_twostates_neg",
        "success_count": n_success
    })

    # plot_scatter_motifrmsd(agg, motif).show()

df_neg = pd.DataFrame(successes_neg)
(df_neg["success_count"]!=0).sum()


The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


10

In [8]:
len(df_neg[df_neg["success_count"]!=0])

10

### plot all (onemotif_twostates_pos)

In [6]:
# out_dir = f"./out/onemotif_twostates_pos/"
out_dir = f"out/onemotif_twostates_pos_OQO/"
successes_pos = []

for motif in sorted(os.listdir(out_dir)):
    motif_out_dir = os.path.join(out_dir, motif)

    agg_path = os.path.join(motif_out_dir, "_aggresults.csv")
    full_path = os.path.join(motif_out_dir, "_fullresults.csv")

    if not (os.path.exists(agg_path) and os.path.exists(full_path)):
        motif_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motif}.pdb"
        full, agg = motif_results(motif_out_dir, motif_pdb, motif)
    else:
        full = pd.read_csv(full_path)
        agg = pd.read_csv(agg_path)

    # condition for one motif two states positive allostery
    n_success = len(
        agg[(agg["motifRMSD_mean_unbound"] > 1.0) 
            & (agg["motifRMSD_mean_bound"] <= 1.0) 
            & ((agg["motifRMSD_mean_bound"] - agg["motifRMSD_mean_unbound"]).abs() >= 0.5)
            & (agg["motifRMSD_std_unbound"] <= 0.5)
            & (agg["motifRMSD_std_bound"] <= 0.5)
            ]
    )

    successes_pos.append({
        "motif": motif,
        "effector": "OQO",
        "task": "onemotif_twostates_pos", 
        "success_count": n_success
    })

    # plot_scatter_motifrmsd(agg, motif).show()

df_pos = pd.DataFrame(successes_pos)

(df_pos["success_count"]!=0).sum()

8

### one motif two states neg with zinc modulation

In [7]:
successes_neg_zn = []
out_dir = f"out/onemotif_twostates_neg_zn/"

for motif in sorted(os.listdir(out_dir)):
    motif_out_dir = os.path.join(out_dir, motif)
    
    agg_path = os.path.join(motif_out_dir, "_aggresults.csv")
    full_path = os.path.join(motif_out_dir, "_fullresults.csv")

    if not (os.path.exists(agg_path) and os.path.exists(full_path)):
        motif_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motif}.pdb"
        full, agg = motif_results(motif_out_dir, motif_pdb, motif)
    else:
        full = pd.read_csv(full_path)
        agg = pd.read_csv(agg_path)

    # condition for one motif two states negative allostery
    n_success = len(
        agg[(agg["motifRMSD_mean_unbound"] <= 1.0) 
            & (agg["motifRMSD_mean_bound"] > 1.0) 
            &  ((agg["motifRMSD_mean_bound"] - agg["motifRMSD_mean_unbound"]).abs() >= 0.5)
            & (agg["motifRMSD_std_unbound"] <= 0.5)
            & (agg["motifRMSD_std_bound"] <= 0.5)
            ]
    )
    successes_neg_zn.append({
        "motif": motif,
        "effector": "[Zn+2]",
        "task": "onemotif_twostates_neg",
        "success_count": n_success
    })

    # plot_scatter_motifrmsd(agg, motif).show()

df_neg_zn = pd.DataFrame(successes_neg_zn)
(df_neg_zn["success_count"]!=0).sum()

6

### one motif two states pos with zinc modulation

In [8]:
# out_dir = f"./out/onemotif_twostates_pos/"
out_dir = f"out/onemotif_twostates_pos_zn/"
successes_pos_zn = []

for motif in sorted(os.listdir(out_dir)):
    motif_out_dir = os.path.join(out_dir, motif)

    agg_path = os.path.join(motif_out_dir, "_aggresults.csv")
    full_path = os.path.join(motif_out_dir, "_fullresults.csv")

    if not (os.path.exists(agg_path) and os.path.exists(full_path)):
        motif_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motif}.pdb"
        full, agg = motif_results(motif_out_dir, motif_pdb, motif)
    else:
        full = pd.read_csv(full_path)
        agg = pd.read_csv(agg_path)

    # condition for one motif two states positive allostery
    n_success = len(
        agg[(agg["motifRMSD_mean_unbound"] > 1.0) 
            & (agg["motifRMSD_mean_bound"] <= 1.0) 
            & ((agg["motifRMSD_mean_bound"] - agg["motifRMSD_mean_unbound"]).abs() >= 0.5)
            & (agg["motifRMSD_std_unbound"] <= 0.5)
            & (agg["motifRMSD_std_bound"] <= 0.5)
            ] 
    )

    successes_pos_zn.append({
        "motif": motif,
        "effector": "[Zn+2]",
        "task": "onemotif_twostates_pos", 
        "success_count": n_success
    })

    # plot_scatter_motifrmsd(agg, motif).show()

df_pos_zn = pd.DataFrame(successes_pos_zn)
(df_pos_zn["success_count"]!=0).sum()

7

### 

In [9]:
successes_neg_mg = []
out_dir = f"out/onemotif_twostates_neg_mg/"

for motif in sorted(os.listdir(out_dir)):
    motif_out_dir = os.path.join(out_dir, motif)
    
    agg_path = os.path.join(motif_out_dir, "_aggresults.csv")
    full_path = os.path.join(motif_out_dir, "_fullresults.csv")

    if not (os.path.exists(agg_path) and os.path.exists(full_path)):
        motif_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motif}.pdb"
        full, agg = motif_results(motif_out_dir, motif_pdb, motif)
    else:
        full = pd.read_csv(full_path)
        agg = pd.read_csv(agg_path)

    # condition for one motif two states negative allostery
    n_success = len(
        agg[(agg["motifRMSD_mean_unbound"] <= 1.0) 
            & (agg["motifRMSD_mean_bound"] > 1.0)  
            & ((agg["motifRMSD_mean_bound"] - agg["motifRMSD_mean_unbound"]).abs() >= 0.5)
            & (agg["motifRMSD_std_unbound"] <= 0.5)
            & (agg["motifRMSD_std_bound"] <= 0.5)
            ]
    )
    
    successes_neg_mg.append({
        "motif": motif,
        "effector":"[Mg+2]",
        "task": "onemotif_twostates_neg",
        "success_count": n_success
    })

    # plot_scatter_motifrmsd(agg, motif).show()

df_neg_mg = pd.DataFrame(successes_neg_mg)
(df_neg_mg["success_count"]!=0).sum()


5

In [10]:
# out_dir = f"./out/onemotif_twostates_pos/"
out_dir = f"out/onemotif_twostates_pos_mg/"
successes_pos_mg = []

for motif in sorted(os.listdir(out_dir)):
    motif_out_dir = os.path.join(out_dir, motif)

    agg_path = os.path.join(motif_out_dir, "_aggresults.csv")
    full_path = os.path.join(motif_out_dir, "_fullresults.csv")

    if not (os.path.exists(agg_path) and os.path.exists(full_path)):
        motif_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motif}.pdb"
        full, agg = motif_results(motif_out_dir, motif_pdb, motif)
    else:
        full = pd.read_csv(full_path)
        agg = pd.read_csv(agg_path)

    # condition for one motif two states positive allostery
    n_success = len(
        agg[(agg["motifRMSD_mean_unbound"] > 1.0) 
            & (agg["motifRMSD_mean_bound"] <= 1.0) 
            & ((agg["motifRMSD_mean_bound"] - agg["motifRMSD_mean_unbound"]).abs() >= 0.5)
            & (agg["motifRMSD_std_unbound"] <= 0.5)
            & (agg["motifRMSD_std_bound"] <= 0.5)
            ] 
    )

    successes_pos_mg.append({
        "motif": motif,
        "effector": "[Mg+2]",
        "task": "onemotif_twostates_pos", 
        "success_count": n_success
    })

    # plot_scatter_motifrmsd(agg, motif).show()

df_pos_mg = pd.DataFrame(successes_pos_mg)
(df_pos_mg["success_count"]!=0).sum()


6

In [11]:
successes_neg_fad = []
out_dir = f"out/onemotif_twostates_neg_FAD/"

for motif in sorted(os.listdir(out_dir)):
    motif_out_dir = os.path.join(out_dir, motif)
    
    agg_path = os.path.join(motif_out_dir, "_aggresults.csv")
    full_path = os.path.join(motif_out_dir, "_fullresults.csv")

    if not (os.path.exists(agg_path) and os.path.exists(full_path)):
        motif_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motif}.pdb"
        full, agg = motif_results(motif_out_dir, motif_pdb, motif)
    else:
        full = pd.read_csv(full_path)
        agg = pd.read_csv(agg_path)

    # condition for one motif two states negative allostery
    n_success = len(
        agg[(agg["motifRMSD_mean_unbound"] <= 1.0) 
            & (agg["motifRMSD_mean_bound"] > 1.0)  
            & ((agg["motifRMSD_mean_bound"] - agg["motifRMSD_mean_unbound"]).abs() >= 0.5)
            & (agg["motifRMSD_std_unbound"] <= 0.5)
            & (agg["motifRMSD_std_bound"] <= 0.5)
            ]
    )
    
    successes_neg_fad.append({
        "motif": motif,
        "effector":"FAD",
        "task": "onemotif_twostates_neg",
        "success_count": n_success
    })

    # plot_scatter_motifrmsd(agg, motif).show()

df_neg_fad = pd.DataFrame(successes_neg_fad)
(df_neg_fad["success_count"]!=0).sum()



7

In [ ]:
# out_dir = f"./out/onemotif_twostates_pos/"
out_dir = f"out/onemotif_twostates_pos_FAD/"
successes_pos_FAD = []

for motif in sorted(os.listdir(out_dir)):
    motif_out_dir = os.path.join(out_dir, motif)

    agg_path = os.path.join(motif_out_dir, "_aggresults.csv")
    full_path = os.path.join(motif_out_dir, "_fullresults.csv")

    if not (os.path.exists(agg_path) and os.path.exists(full_path)):
        motif_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motif}.pdb"
        full, agg = motif_results(motif_out_dir, motif_pdb, motif)
    else:
        full = pd.read_csv(full_path)
        agg = pd.read_csv(agg_path)

    # condition for one motif two states positive allostery
    n_success = len(
        agg[(agg["motifRMSD_mean_unbound"] > 1.0) 
            & (agg["motifRMSD_mean_bound"] <= 1.0) 
            & ((agg["motifRMSD_mean_bound"] - agg["motifRMSD_mean_unbound"]).abs() >= 0.5)
            & (agg["motifRMSD_std_unbound"] <= 0.5)
            & (agg["motifRMSD_std_bound"] <= 0.5)
            ] 
    )

    successes_pos_FAD.append({
        "motif": motif,
        "effector": "FAD",
        "task": "onemotif_twostates_pos", 
        "success_count": n_success
    })

    # plot_scatter_motifrmsd(agg, motif).show()

df_pos_FAD = pd.DataFrame(successes_pos_FAD)
(df_pos_FAD["success_count"]!=0).sum()


7

### DNA

In [4]:
successes_neg_ecoridna = []
out_dir = f"out/onemotif_twostates_neg_EcoRIdna/"

for motif in sorted(os.listdir(out_dir)):
    motif_out_dir = os.path.join(out_dir, motif)
    
    agg_path = os.path.join(motif_out_dir, "_aggresults.csv")
    full_path = os.path.join(motif_out_dir, "_fullresults.csv")

    if not (os.path.exists(agg_path) and os.path.exists(full_path)):
        motif_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motif}.pdb"
        full, agg = motif_results(motif_out_dir, motif_pdb, motif)
    else:
        full = pd.read_csv(full_path)
        agg = pd.read_csv(agg_path)

    # condition for one motif two states negative allostery
    n_success = len(
        agg[(agg["motifRMSD_mean_unbound"] <= 1.0) 
            & (agg["motifRMSD_mean_bound"] > 1.0)  
            & ((agg["motifRMSD_mean_bound"] - agg["motifRMSD_mean_unbound"]).abs() >= 0.5)
            & (agg["motifRMSD_std_unbound"] <= 0.5)
            & (agg["motifRMSD_std_bound"] <= 0.5)
            ]
    )
    
    successes_neg_ecoridna.append({
        "motif": motif,
        "effector":"DNA",
        "task": "onemotif_twostates_neg",
        "success_count": n_success
    })

    # plot_scatter_motifrmsd(agg, motif).show()

df_neg_dna = pd.DataFrame(successes_neg_ecoridna)
(df_neg_dna["success_count"]!=0).sum()



6

In [17]:
out_dir = f"out/onemotif_twostates_pos_EcoRIdna/"
successes_pos_ecoridna = []

for motif in sorted(os.listdir(out_dir)):
    motif_out_dir = os.path.join(out_dir, motif)

    agg_path = os.path.join(motif_out_dir, "_aggresults.csv")
    full_path = os.path.join(motif_out_dir, "_fullresults.csv")

    if not (os.path.exists(agg_path) and os.path.exists(full_path)):
        motif_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motif}.pdb"
        full, agg = motif_results(motif_out_dir, motif_pdb, motif)
    else:
        full = pd.read_csv(full_path)
        agg = pd.read_csv(agg_path)

    # condition for one motif two states positive allostery
    n_success = len(
        agg[(agg["motifRMSD_mean_unbound"] > 1.0) 
            & (agg["motifRMSD_mean_bound"] <= 1.0) 
            & ((agg["motifRMSD_mean_bound"] - agg["motifRMSD_mean_unbound"]).abs() >= 0.5)
            & (agg["motifRMSD_std_unbound"] <= 0.5)
            & (agg["motifRMSD_std_bound"] <= 0.5)
            ] 
    )

    successes_pos_ecoridna.append({
        "motif": motif,
        "effector": "DNA",
        "task": "onemotif_twostates_pos", 
        "success_count": n_success
    })

    # plot_scatter_motifrmsd(agg, motif).show()

df_pos_dna = pd.DataFrame(successes_pos_ecoridna)
(df_pos_dna["success_count"]!=0).sum()


7

### aggregate all onemotif two states

In [14]:
import plotly.express as px

# Combine results
df_all = pd.concat(
    [df_pos, df_pos_FAD, df_pos_zn, df_pos_mg, df_pos_dna,
     df_neg, df_neg_fad, df_neg_zn, df_neg_mg, df_neg_dna],
    ignore_index=True
)
df_all["type"] = df_all["task"].apply(lambda x: "pos" if "pos" in x else "neg")
nonzero_motifs = df_all.groupby("motif")["success_count"].sum()
nonzero_motifs = nonzero_motifs[nonzero_motifs > 0].index
df_all = df_all[df_all["motif"].isin(nonzero_motifs)]

df_pos_plot = df_all[df_all["type"] == "pos"]
fig_pos = px.bar(
    df_pos_plot,
    x="motif",
    y="success_count",
    color="effector",
    barmode="group",
    title="Positive allostery",
    labels={"success_count": "Successes", "motif": "Motif"},
    color_discrete_map={"OQO": "teal", "FAD": "green", "Zn": "orange", "Mg": "blue"}
)
fig_pos.update_layout(
    font=dict(size=14, color="black"),
    xaxis=dict(tickangle=45)
)

df_neg_plot = df_all[df_all["type"] == "neg"]
fig_neg = px.bar(
    df_neg_plot,
    x="motif",
    y="success_count",
    color="effector",
    barmode="group",
    title="Negative allostery",
    labels={"success_count": "Successes", "motif": "Motif"},
    color_discrete_map={"OQO": "teal", "FAD": "green", "Zn": "orange", "Mg": "blue"}
)
fig_neg.update_layout(
    font=dict(size=14, color="black"),
    xaxis=dict(tickangle=45)
)

fig_pos.show()
fig_neg.show()


In [18]:
effector_category = {
    "OQO": "ligand",
    "FAD": "ligand",
    "Zn": "ion",
    "Mg": "ion"
}
def sort_key(idx):
    eff, typ = idx.split()
    cat = effector_category.get(eff, "other")
    return (0 if typ=="neg" else 1, 
            0 if cat=="ligand" else 1, 
            eff)

def shorten_label(label: str) -> str:
    if label.endswith("_short"):
        return label.replace("_short", "(s)")
    elif label.endswith("_med"):
        return label.replace("_med", "(m)")
    elif label.endswith("_long"):
        return label.replace("_long", "(l)")
    return label  # leave unchanged


df_all = pd.concat([df_pos,df_pos_FAD, df_pos_zn,df_pos_mg,df_pos_dna, df_neg, df_neg_fad, df_neg_zn,df_neg_mg, df_neg_dna], ignore_index=True)
df_all["type"] = df_all["task"].apply(lambda x: "pos" if "pos" in x else "neg")
heatmap_df = df_all.pivot_table(
    index=["effector", "type"],
    columns="motif",
    values="success_count",
    fill_value=0
)

heatmap_df.index = heatmap_df.index.map(lambda x: f"{x[1]} {x[0]}   ")
heatmap_df = heatmap_df.loc[:, (heatmap_df != 0).any(axis=0)]
heatmap_df = heatmap_df.loc[
    sorted(heatmap_df.index, key=lambda s: (s.split()[0], s.split()[1]))[::-1]
]

neg_rows = [i for i, idx in enumerate(heatmap_df.index) if "neg" in idx]
x_labels = [shorten_label(col) for col in heatmap_df.columns]

fig = px.imshow(
    heatmap_df.values,
    x=x_labels,
    y=heatmap_df.index,
    color_continuous_scale="Blues",
    aspect="auto",
    text_auto=True
)

fig.update_layout(
    # title="One motif two states successes",
    font=dict( size=16,color="black"),
    xaxis=dict(side="bottom", tickangle=90,ticklabelstandoff=10),   
    yaxis=dict(autorange="reversed"),
    # xaxis_title="Motif",
    # yaxis_title="Effector",
    coloraxis_colorbar=dict(title="Successes", orientation="h"),
    width=900,
    height=800,
    margin=dict(l=50, r=50, t=50, b=100),
    coloraxis_colorbar_y=-0.25,
    coloraxis_colorbar_x=30
)


fig.add_hline(
    y=len(heatmap_df.index)//2 -0.5 ,
    line_width=2,
    line_dash="solid",
    line_color="black"
)
fig.add_shape(
    type="rect",
    xref="paper", yref="paper", 
    x0=0, y0=0, x1=1.0, y1=1.0, 
    line=dict(
        color="black",  
        width=3,       
    ),
    layer="below"
)
    
fig.show()

### Main figure pymol selections

In [35]:
motif = "1ycr"
effector = "EcoRIdna"
allostery = "neg"
agg = pd.read_csv( f"out/onemotif_twostates_{allostery}_{effector}/{motif}/_aggresults.csv")

if allostery == "pos":
    res = agg[(agg["motifRMSD_mean_unbound"] > 1.0) & (agg["motifRMSD_mean_bound"] <= 1.0) & ((agg["motifRMSD_mean_bound"] - agg["motifRMSD_mean_unbound"]).abs() >= 0.5) ] 
else:
    res = agg[(agg["motifRMSD_mean_unbound"] <= 1.0) & (agg["motifRMSD_mean_bound"] > 1.0) & ((agg["motifRMSD_mean_bound"] - agg["motifRMSD_mean_unbound"]).abs() >= 0.5) ] 

res # design91

,design,motifRMSD_mean_unbound,motifRMSD_mean_bound,motifRMSD_std_unbound,motifRMSD_std_bound,plddt_unbound,plddt_bound,ptm_unbound,ptm_bound
1,design1,0.920604,2.590008,0.086577,0.140735,0.769835,0.667556,0.358750,0.502190
13,design20,0.795079,1.537295,0.067294,0.596747,0.716375,0.608852,0.329608,0.455515
15,design22,0.896685,1.513270,0.212678,0.738780,0.739354,0.645985,0.332530,0.461259
20,design27,0.888150,3.352578,0.075520,0.016343,0.784391,0.758262,0.345086,0.712041
23,design3,0.819256,3.273531,0.101182,0.093604,0.746162,0.733341,0.413798,0.582081
38,design43,0.814990,1.514873,0.021839,0.079639,0.752485,0.729150,0.565697,0.703778
50,design54,0.930696,2.317985,0.146235,0.041260,0.684183,0.691342,0.275135,0.645062
56,design6,0.841163,1.703094,0.055740,0.039723,0.734644,0.719490,0.418626,0.548437
58,design61,0.820617,1.428848,0.040176,0.520054,0.713759,0.701388,0.344185,0.476052
62,design65,0.977643,1.553063,0.181841,0.402383,0.673167,0.558441,0.294468,0.392797


In [20]:
import pickle
import numpy as np
with open("/data/cb/mihirb14/projects/BoltzDesign1/out/onemotif_twostates_neg_EcoRIdna/1qjg/design8/1qjg_spec.pkl","rb") as f:
    motif_spec = pickle.load(f)
np.where(motif_spec["motif_mask"] == True)
# motif_spec

(array([17, 42, 64]),)

In [ ]:
motif = "1prw"
effector = "FAD"
allostery = "neg"
agg = pd.read_csv( f"out/onemotif_twostates_{allostery}_{effector}/{motif}/_aggresults.csv")

if allostery == "pos":
    res = agg[(agg["motifRMSD_mean_unbound"] > 1.0) & (agg["motifRMSD_mean_bound"] <= 1.0) & ((agg["motifRMSD_mean_bound"] - agg["motifRMSD_mean_unbound"]).abs() >= 0.5) ] 
else:
    res = agg[(agg["motifRMSD_mean_unbound"] <= 1.0) & (agg["motifRMSD_mean_bound"] > 1.0) & ((agg["motifRMSD_mean_bound"] - agg["motifRMSD_mean_unbound"]).abs() >= 0.5) ] 

res # chose design 43

,design,motifRMSD_mean_unbound,motifRMSD_mean_bound,motifRMSD_std_unbound,motifRMSD_std_bound,plddt_unbound,plddt_bound,ptm_unbound,ptm_bound
26,design32,0.925166,10.489433,0.234081,5.462654,0.653396,0.544922,0.451686,0.386196
28,design34,0.849516,1.495307,0.049608,0.114728,0.698914,0.600055,0.647119,0.606558
38,design43,0.649634,1.596919,0.098145,0.101167,0.822414,0.699252,0.671142,0.778462
85,design86,0.621368,4.321555,0.022866,0.518978,0.822285,0.478172,0.707584,0.408618


In [ ]:
motif = "2kl8"
effector = "OQO"
allostery = "pos"
agg = pd.read_csv( f"out/onemotif_twostates_{allostery}_{effector}/{motif}/_aggresults.csv")

if allostery == "pos":
    res = agg[(agg["motifRMSD_mean_unbound"] > 1.0) & (agg["motifRMSD_mean_bound"] <= 1.0) & ((agg["motifRMSD_mean_bound"] - agg["motifRMSD_mean_unbound"]).abs() >= 0.5) ] 
else:
    res = agg[(agg["motifRMSD_mean_unbound"] <= 1.0) & (agg["motifRMSD_mean_bound"] > 1.0) & ((agg["motifRMSD_mean_bound"] - agg["motifRMSD_mean_unbound"]).abs() >= 0.5) ] 

res # chose deisgn 85

,design,motifRMSD_mean_unbound,motifRMSD_mean_bound,motifRMSD_std_unbound,motifRMSD_std_bound,plddt_unbound,plddt_bound,ptm_unbound,ptm_bound
1,design1,1.687263,0.961682,0.067936,0.058059,0.748070,0.779700,0.710269,0.865131
21,design28,6.081819,0.863795,0.631722,0.056859,0.732617,0.689849,0.647894,0.742394
47,design51,3.695004,0.903991,0.291611,0.072796,0.581448,0.673580,0.346060,0.757436
84,design85,8.406400,0.952696,0.469640,0.080004,0.737670,0.657541,0.570603,0.660501
89,design9,1.608672,0.948357,0.229986,0.057169,0.719392,0.877177,0.634836,0.921576


In [51]:
motif = "3ixt"
effector = "mg"
allostery = "pos"
agg = pd.read_csv( f"out/onemotif_twostates_{allostery}_{effector}/{motif}/_aggresults.csv")

if allostery == "pos":
    res = agg[(agg["motifRMSD_mean_unbound"] > 1.0) & (agg["motifRMSD_mean_bound"] <= 1.0) & ((agg["motifRMSD_mean_bound"] - agg["motifRMSD_mean_unbound"]).abs() >= 0.5) ] 
else:
    res = agg[(agg["motifRMSD_mean_unbound"] <= 1.0) & (agg["motifRMSD_mean_bound"] > 1.0) & ((agg["motifRMSD_mean_bound"] - agg["motifRMSD_mean_unbound"]).abs() >= 0.5) ] 

res

,design,motifRMSD_mean_unbound,motifRMSD_mean_bound,motifRMSD_std_unbound,motifRMSD_std_bound,plddt_unbound,plddt_bound,ptm_unbound,ptm_bound
3,design11,7.515732,0.507741,1.194956,0.044439,0.617627,0.647528,0.395262,0.585052
11,design19,7.399507,0.916375,0.855732,0.201025,0.552770,0.649526,0.177871,0.548991
18,design25,5.198252,0.863946,2.191748,0.196159,0.623011,0.619823,0.214540,0.383892
30,design36,6.213734,0.716814,1.157821,0.388461,0.604938,0.627159,0.252673,0.516514
36,design41,4.007356,0.732864,1.636691,0.272087,0.651797,0.654632,0.239881,0.592582
37,design42,2.551097,0.883363,0.629937,0.255979,0.583657,0.620277,0.222364,0.467988
45,design5,4.661182,0.395812,2.032678,0.042729,0.594705,0.782123,0.202426,0.771153
46,design50,5.994926,0.709129,2.802440,0.061634,0.666039,0.725384,0.301777,0.596408
50,design54,5.659430,0.676143,2.780285,0.115131,0.622800,0.610573,0.221339,0.495254
59,design62,4.188784,0.924784,0.645035,0.170123,0.646291,0.766800,0.239168,0.660170


In [123]:
full = pd.read_csv( f"out/onemotif_twostates_pos_FAD/{motif}/_fullresults.csv")
full


,design,state,sample,motifrmsd,plddt,ptm
0,design0,0,0,4.648955,0.688689,0.340639
1,design0,0,1,3.559667,0.696597,0.334436
2,design0,0,2,3.440545,0.706353,0.327854
3,design0,0,3,4.181741,0.693394,0.347819
4,design0,0,4,3.392759,0.698841,0.337400
...,...,...,...,...,...,...
995,design99,1,0,3.518249,0.527769,0.567354
996,design99,1,1,3.208068,0.501812,0.434723
997,design99,1,2,3.350734,0.504909,0.434333
998,design99,1,3,2.993417,0.534420,0.582901


### two motifs two states

In [123]:
# out_dir = f"./out/twomotif_twostates/"
# out_dir = f"./out/twomoitf_twostates_mg/"
out_dir = f"./out/twomotif_twostates_twoligands/"
# out_dir = "./out/higherantimotif/twomotif_twostates/"

motifA = "3ixt" # active in state 0 when ligand unbound, inactive in state 1 when ligand bound
motifB = "1ycr" # inactive in state 0 when ligand unbound, active in state 1 when ligand bound
# motifB = "6e6r_long" # inactive in state 0 when ligand unbound, active in state 1 when ligand bound

motifA_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motifA}.pdb"
motifB_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motifB}.pdb"

motif_out_dir = os.path.join(out_dir,f"{motifA}_{motifB}")
print(motif_out_dir)

./out/twomotif_twostates_twoligands/3ixt_1ycr


In [124]:
motifA_df_full,motifA_df_agg = motif_results(motif_out_dir,motifA_pdb,motifA)
display(plot_scatter_motifrmsd(motifA_df_agg,motifA))
motifB_df_full,motifB_df_agg = motif_results(motif_out_dir,motifB_pdb,motifB)
plot_scatter_motifrmsd(motifB_df_agg,motifB)


In [125]:
success_A = motifA_df_agg[(motifA_df_agg["motifRMSD_mean_unbound"] <= 1.0) & (motifA_df_agg["motifRMSD_mean_bound"] > 1.0) & ((motifA_df_agg["motifRMSD_mean_bound"] - motifA_df_agg["motifRMSD_mean_unbound"]).abs() >= 0.5)] # motif A active in state 0, inactive in state 1
success_B = motifB_df_agg[(motifB_df_agg["motifRMSD_mean_unbound"] > 1.0) & (motifB_df_agg["motifRMSD_mean_bound"] <= 1.0)& ((motifB_df_agg["motifRMSD_mean_bound"] - motifB_df_agg["motifRMSD_mean_unbound"]).abs() >= 0.5)] # motif B inactive in state 0, active in state 1
display(success_A)
display(success_B)

,design,motifRMSD_mean_unbound,motifRMSD_mean_bound,motifRMSD_std_unbound,motifRMSD_std_bound,plddt_unbound,plddt_bound,ptm_unbound,ptm_bound
19,design26,0.732228,2.289161,0.428667,0.152670,0.537747,0.667688,0.386718,0.533921
31,design37,0.627315,8.494970,0.066834,0.158828,0.802445,0.882553,0.820833,0.828194
35,design40,0.591126,3.585539,0.056860,0.332936,0.642317,0.775785,0.639481,0.762727
44,design49,0.538781,4.211472,0.045860,1.322111,0.558503,0.592559,0.510093,0.496486
45,design5,0.846695,2.042007,0.104683,1.642993,0.466070,0.499519,0.353565,0.336049
46,design50,0.936779,1.803149,0.057379,0.063803,0.670047,0.783040,0.602673,0.661657
56,design6,0.672096,5.875584,0.182406,0.261112,0.635796,0.710766,0.589842,0.513068
64,design67,0.453953,1.569655,0.083305,0.208953,0.607552,0.704142,0.653571,0.677850
79,design80,0.594838,2.144167,0.070519,0.476718,0.817179,0.566659,0.816756,0.378212


,design,motifRMSD_mean_unbound,motifRMSD_mean_bound,motifRMSD_std_unbound,motifRMSD_std_bound,plddt_unbound,plddt_bound,ptm_unbound,ptm_bound
28,design34,1.421262,0.679071,0.55397,0.07218,0.559379,0.783833,0.393688,0.643424


In [126]:
set(success_A["design"]).intersection(set(success_B["design"]))

set()

In [58]:
motifB_df_full[motifB_df_full["design"]=="design1"]

,design,state,sample,motifrmsd,plddt,ptm
10,design1,0,0,0.727359,0.679814,0.502158
11,design1,0,1,1.741815,0.710819,0.514460
12,design1,0,2,1.646023,0.709005,0.503220
13,design1,0,3,0.902187,0.684233,0.485717
14,design1,0,4,0.769684,0.672425,0.417673
15,design1,1,0,0.658855,0.767230,0.749349
16,design1,1,1,0.693115,0.755146,0.710810
17,design1,1,2,0.750947,0.776669,0.743204
18,design1,1,3,0.777191,0.758175,0.723047
19,design1,1,4,0.765911,0.774413,0.743548


In [119]:
motifA_df_full[motifA_df_full["design"]=="design91"]

,design,state,sample,motifrmsd,plddt,ptm
910,design91,0,0,0.390635,0.684185,0.589732
911,design91,0,1,0.399948,0.688616,0.620299
912,design91,0,2,0.382589,0.688305,0.585404
913,design91,0,3,0.363413,0.688271,0.584492
914,design91,0,4,0.414448,0.694405,0.612431
915,design91,1,0,1.196500,0.781649,0.802171
916,design91,1,1,1.286581,0.789073,0.805061
917,design91,1,2,1.261281,0.795580,0.803629
918,design91,1,3,1.275448,0.787099,0.794050
919,design91,1,4,1.203851,0.783332,0.785155


In [120]:
motifB_df_full[motifB_df_full["design"]=="design91"]

,design,state,sample,motifrmsd,plddt,ptm
910,design91,0,0,1.514383,0.684185,0.589732
911,design91,0,1,1.265376,0.688616,0.620299
912,design91,0,2,1.426285,0.688305,0.585404
913,design91,0,3,1.468707,0.688271,0.584492
914,design91,0,4,1.311683,0.694405,0.612431
915,design91,1,0,1.466845,0.781649,0.802171
916,design91,1,1,0.635651,0.789073,0.805061
917,design91,1,2,0.679535,0.795580,0.803629
918,design91,1,3,0.587565,0.787099,0.794050
919,design91,1,4,0.660044,0.783332,0.785155
